# Exercise 5 - Mapping High Altitude Rivers

In this exercise, we will use Google Earth Engine to explore changes in high-altitude rivers. Climate change has had a strong impact on glaciers and snow cover, which has changed how water moves from the highest peaks down to rivers. This has had impacts on the timing of water availability, the temperature of the water, and also how much sediment it transports. More frequent landslides due to intense precipitation and new high-elevation infrastructure also have strong impacts on rivers.

### Goals

As many high-elevation rivers are quite thin, we cannot use many climate model and satellite data sets to examine them. We can, however, use Landsat data, which as a 30 m pixel size, to look at thin rivers. We will use Landsat for two purposes: (1) river temperature and (2) river color, which is a proxy for sediment content.

Let's start by adding some python modules:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy
import cartopy.crs as ccrs
import datetime
from shapely.geometry import mapping

In [ ]:
import ee
import geemap
#ee.Authenticate()
ee.Initialize()
print(ee.__version__)

The first thing we need to do is identify where rivers actually are. There are two main approaches we could try: (1) use a water mask, such as this one: [https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater) or use a flow network, such as this one: [https://www.hydrosheds.org/products/hydrorivers](https://www.hydrosheds.org/products/hydrorivers). 

The water mask approach will give us a pixel-wise analysis, whereas if we use vector (flow network) data, we can have more control over how we calculate trends and process the data. 

Let's try the grid-based approach first, as we can do this using only Earth Engine code. We first want to define a water mask -- we are only interested in water pixels for this analsyis. The [JRC](https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_4_GlobalSurfaceWater) data is a nice data set to use for this purpose. We can even define a water occurence threshold to focus on areas that are most often covered in water:

In [ ]:
water_occ = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
water_threshold = 25 #Choose the threshold for how often a pixel is 'water'
mask = water_occ.gt(water_threshold) #Update the mask with our threshold
water_mask = water_occ.updateMask(mask)

We can try visualizing the water mask as a test. We will look just downstream of the glacier lake we examined this morning:

In [ ]:
river_center = ee.Geometry.Point([85.21, 28.07]) 

visualization = {'bands': ['occurrence'], 'palette':['3B4CC0', '6F91F2', 'A9C5FC', 'DDDDDD', 'F6B69B', 'E6745B', 'B40426'], 'min':water_threshold, 'max': 100, 'alpha':0.75}
Map = geemap.Map(center=[28.07, 85.21], zoom=14) 
url = "http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}"
Map.add_tile_layer(url, name="Google Map", attribution="Google") #Add Google Earth imagery as a background
Map.addLayer(water_mask, visualization)
Map.addLayer(river_center) #Plot the center point

#Red is frequent water cover, blue is less frequent
Map

Try modifying the water mask threshold to see what looks best for identifying river pixels. We can be more or less conservative depending on how low the allowed 'water percentage' is. 

Once we have our mask, we can then get Landsat data. We need to run several processing steps to conver the data to surface temperature, mask out clouds, etc.

In [ ]:
def prep457bands(img):
    systime = img.get('system:time_start')
    elev = img.get("SUN_ELEVATION")
    sza = ee.Number(90).subtract(elev)

    #L4-7: B1=Blue, B2=Green, B3=Red, B4=NIR, B5=SWIR1, B7=SWIR2
    optical = img.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'], ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']).multiply(0.0000275).add(-0.2)
    thermal = img.select(["ST_B6"],["surface_temp"]).multiply(0.00341802).add(149).add(-273.15)
    
    qa = img.select(["QA_PIXEL"])
    cloud1 = qa.bitwiseAnd(2).eq(0)       # bit 1, dilated cloud
    cloud3 = qa.bitwiseAnd(8).eq(0)       # bit 3, cloud
    cloudshadow = qa.bitwiseAnd(16).eq(0) # bit 4, cloud shadow
    snow = qa.bitwiseAnd(32).eq(0)        # bit 5, snow
    cloud_confid = qa.rightShift(8).bitwiseAnd(3).lt(2)   # bits 8-9
    cloudsh_confid = qa.rightShift(10).bitwiseAnd(3).lt(2) # bits 10-11
    snow_confid = qa.rightShift(12).bitwiseAnd(3).lt(2)   # bits 12-13
    cirrus_confid = qa.rightShift(14).bitwiseAnd(3).lt(2) # bits 14-15 (L7 only usually)
    
    full_image = optical.addBands(thermal)

    #Apply all masks
    updated = full_image.updateMask(cloud1)\
        .updateMask(cloud3)\
        .updateMask(snow)\
        .updateMask(cloudshadow)\
        .updateMask(cloud_confid)\
        .updateMask(snow_confid)\
        .updateMask(cirrus_confid)\
        .updateMask(cloudsh_confid)

    return ee.Image(updated).copyProperties(img).set("system:time_start",systime).set("SOLAR_ZENITH_ANGLE", sza)

def prep89bands(img):
    systime = img.get('system:time_start')
    elev = img.get("SUN_ELEVATION")
    sza = ee.Number(90).subtract(elev)

    #L8-9: B2=Blue, B3=Green, B4=Red, B5=NIR, B6=SWIR1, B7=SWIR2
    optical = img.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']).multiply(0.0000275).add(-0.2)
    thermal = img.select(["ST_B10"],["surface_temp"]).multiply(0.00341802).add(149).add(-273.15)

    qa = img.select(["QA_PIXEL"])
    cloud1 = qa.bitwiseAnd(2).eq(0)  # bit 1, dilated cloud
    cloud2 = qa.bitwiseAnd(4).eq(0)  # bit 2, cirrus
    cloud3 = qa.bitwiseAnd(8).eq(0)  # bit 3, cloud
    cloudshadow = qa.bitwiseAnd(16).eq(0) # bit 4, cloud shadow
    snow = qa.bitwiseAnd(32).eq(0)  # bit 5, snow
    cloud_confid = qa.rightShift(8).bitwiseAnd(3).lt(2)
    cloudsh_confid = qa.rightShift(10).bitwiseAnd(3).lt(2)
    snow_confid = qa.rightShift(12).bitwiseAnd(3).lt(2)
    cirrus_confid = qa.rightShift(14).bitwiseAnd(3).lt(2)

    full_image = optical.addBands(thermal)

    updated = full_image.updateMask(cloud1)\
        .updateMask(cloud2)\
        .updateMask(cloud3)\
        .updateMask(cloudshadow)\
        .updateMask(snow)\
        .updateMask(cloud_confid)\
        .updateMask(cloudsh_confid)\
        .updateMask(snow_confid)\
        .updateMask(cirrus_confid)

    return ee.Image(updated).copyProperties(img).set("system:time_start",systime).set("SOLAR_ZENITH_ANGLE", sza)

l5t1 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
l5t2 = ee.ImageCollection("LANDSAT/LT05/C02/T2_L2")
l7t1 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
l7t2 = ee.ImageCollection("LANDSAT/LE07/C02/T2_L2")
l8t1 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
l8t2 = ee.ImageCollection("LANDSAT/LC08/C02/T2_L2")
l9t1 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
l9t2 = ee.ImageCollection("LANDSAT/LC09/C02/T2_L2")

start_date = '1983-10-01'
end_date = '2026-10-01'
scale = 30
rmse = 24 

l5 = l5t1.merge(l5t2).filterMetadata('L1_PROCESSING_LEVEL','equals','L1TP') \
      .filterMetadata('GEOMETRIC_RMSE_MODEL','not_greater_than',rmse) \
      .map(prep457bands, True).filterDate(start_date, end_date)

l7 = l7t1.merge(l7t2).filterMetadata('L1_PROCESSING_LEVEL','equals','L1TP') \
      .filterMetadata('GEOMETRIC_RMSE_MODEL','not_greater_than',rmse) \
      .map(prep457bands, True).filterDate(start_date, end_date)

l8 = l8t1.merge(l8t2).filterMetadata('L1_PROCESSING_LEVEL','equals','L1TP') \
      .filterMetadata('GEOMETRIC_RMSE_MODEL','not_greater_than',rmse) \
      .filterMetadata('TIRS_SSM_POSITION_STATUS','not_equals','SWITCHED') \
      .map(prep89bands, True).filterDate(start_date, end_date)

l9 = l9t1.merge(l9t2).filterMetadata('L1_PROCESSING_LEVEL','equals','L1TP') \
      .filterMetadata('GEOMETRIC_RMSE_MODEL','not_greater_than',rmse) \
      .filterMetadata('TIRS_SSM_POSITION_STATUS','not_equals','SWITCHED') \
      .map(prep89bands, True).filterDate(start_date, end_date)

#Contains bands: blue, green, red, nir, swir1, swir2, surface_temp
full_collection = ee.ImageCollection(l5).merge(l7).merge(l8).merge(l9).sort('system:time_start')

The 'full_collection' now has all of the Landsat 5, 7, 8, and 9 data available. We have only kept some bands, and processed surface temperature in a consistent way. Let's take a look at one image:

In [ ]:
river_center = ee.Geometry.Point([85.21, 28.07]) 
#Filter to select a spring image with minimal snow and clouds
single_image = full_collection.filterBounds(river_center).filterDate('2024-03-01', '2024-04-01').first() 

Map = geemap.Map(center=[28.07, 85.21], zoom=14) 
Map.addLayer(river_center) #Plot the center point

#url = "http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}"
#Map.add_tile_layer(url, name="Google Map", attribution="Google") #Can add Google Earth imagery as a background

visualization = {'bands': ['surface_temp'], 'palette':['3B4CC0', '6F91F2', 'A9C5FC', 'DDDDDD', 'F6B69B', 'E6745B', 'B40426'], 'alpha':0.75}
Map.addLayer(single_image, visualization) #This is the surface temperature image

#visualization = {'bands': ['red', 'green', 'blue'], 'min':0, 'max':0.3} #Choose false-color bands
#Map.addLayer(single_image, visualization) #This is a true-color image to see the underlying data

Map

We can now also mask to only water pixels:

In [ ]:
def mask_water(image):
    return image.updateMask(mask)

water_threshold = 25
water_occ = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
mask = water_occ.gt(water_threshold)

water_collection = full_collection.map(mask_water)

In [ ]:
river_center = ee.Geometry.Point([85.21, 28.07]) 
#Filter to select a spring image with minimal snow and clouds
single_image = water_collection.filterBounds(river_center).filterDate('2024-03-01', '2024-04-01').first() 

Map = geemap.Map(center=[28.07, 85.21], zoom=14) 
Map.addLayer(river_center) #Plot the center point

#url = "http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}"
#Map.add_tile_layer(url, name="Google Map", attribution="Google") #Can add Google Earth imagery as a background

visualization = {'bands': ['surface_temp'], 'palette':['3B4CC0', '6F91F2', 'A9C5FC', 'DDDDDD', 'F6B69B', 'E6745B', 'B40426'], 'alpha':0.75}
Map.addLayer(single_image, visualization) #This is the surface temperature image

#visualization = {'bands': ['red', 'green', 'blue'], 'min':0, 'max':0.3}
#Map.addLayer(single_image, visualization) #This is a true-color image to see the underlying data

#Red is warmer, blue is cooler

Map

This seems to have worked out! We now have water temperature data only along the river. 

Let's now add in some sediment content data as well. We will rely on the [Normalized Difference Turbidity Index](https://step.esa.int/main/wp-content/help/versions/13.0.0/snap-toolboxes/eu.esa.opt.opttbx.radiometric.indices.ui/ndti/NdtiAlgorithmSpecification.html) and a simple ratio of Red to NIR ([see here](https://www.sciencedirect.com/science/article/abs/pii/S0034425797001065)). We can add this to all of our masked data as well:

In [ ]:
def add_turbidity(image):
    #Calculate Normalized Difference Turbidity Index
    ndti = image.normalizedDifference(['red', 'green']).rename('NDTI')
    
    #Calculate Red/NIR ratio
    red_nir_ratio = image.select('red').divide(image.select('nir')).rename('RedNIR')
    return image.addBands([ndti, red_nir_ratio])

water_collection = water_collection.map(add_turbidity)

In [ ]:
river_center = ee.Geometry.Point([85.21, 28.07]) 
#Filter to select a spring image with minimal snow and clouds
single_image = water_collection.filterBounds(river_center).filterDate('2024-03-01', '2024-04-01').first() 

Map = geemap.Map(center=[28.07, 85.21], zoom=14) 
Map.addLayer(river_center) #Plot the center point

#url = "http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}"
#Map.add_tile_layer(url, name="Google Map", attribution="Google") #Can add Google Earth imagery as a background

visualization = {'bands': ['NDTI'], 'palette':['3B4CC0', '6F91F2', 'A9C5FC', 'DDDDDD', 'F6B69B', 'E6745B', 'B40426'], 'alpha':0.75, 'min':0, 'max':0.5}
Map.addLayer(single_image, visualization) #This is the turbidity image

#visualization = {'bands': ['red', 'green', 'blue'], 'min':0, 'max':0.3}
#Map.addLayer(single_image, visualization) #This is a true-color image to see the underlying data

#Red is more sediment, blue is less sediment
Map

With this we can look at both temperature and turbidity along any river in our study area. Let's try this over a larger area -- we can use the Trishuli catchment we looked at earlier for this purpose:

In [ ]:
#trishuli_outline = gpd.read_file('../GeoData/Trishuli.geojson')
trishuli_outline = gpd.read_file('https://raw.githubusercontent.com/tasmi/Workshop_Kathmandu_Feb2026/refs/heads/main/Notebooks/GeoData/Trishuli.geojson')
area_of_interest = ee.Geometry.MultiPolygon(ee.List(mapping(trishuli_outline.geometry[0])['coordinates']))

Instead of just choosing one image, let's select all images over a three month period and get their average:

In [ ]:
#Calculate the three-month average:
single_image = water_collection.filterBounds(area_of_interest).filterDate('2024-02-01', '2024-05-01').reduce(ee.Reducer.mean()) 
cenx, ceny = trishuli_outline.centroid[0].x, trishuli_outline.centroid[0].y

Map = geemap.Map(center=[ceny, cenx], zoom=10)
Map.addLayer(area_of_interest)

visualization = {'bands': ['NDTI_mean'], 'palette':['3B4CC0', '6F91F2', 'A9C5FC', 'DDDDDD', 'F6B69B', 'E6745B', 'B40426'], 'alpha':0.75, 'min':0, 'max':0.5}
Map.addLayer(single_image, visualization)

Map

This is starting to get a bit slower -- when working on large areas with high resolution data, you often need to *export* the data instead of visualizing it directly. We can export long-term means of temperature, NDTI, and Red/NIR ratio for the Trishuli to look at offline:

In [ ]:
def run_export(image, crs, filename, scale, region, maxPixels=1e12, cloud_optimized=True):
    task_config = {'fileNamePrefix': filename,'crs': crs,'scale': scale,'maxPixels': maxPixels, 'fileFormat': 'GeoTIFF', 'formatOptions': {'cloudOptimized': cloud_optimized}, 'region': region,}
    task = ee.batch.Export.image.toDrive(image, filename, **task_config)
    task.start()

In [ ]:
temp_mean = water_collection.filterBounds(area_of_interest).select('surface_temp').reduce(ee.Reducer.mean())
run_export(temp_mean, 'epsg:4326', 'Trishuli_WaterTemp_LTMean', scale=30, region=area_of_interest)

ndti_mean = water_collection.filterBounds(area_of_interest).select('NDTI').reduce(ee.Reducer.mean())
run_export(ndti_mean, 'epsg:4326', 'Trishuli_NDTI_LTMean', scale=30, region=area_of_interest)

rednir_mean = water_collection.filterBounds(area_of_interest).select('RedNIR').reduce(ee.Reducer.mean())
run_export(rednir_mean, 'epsg:4326', 'Trishuli_RedNIR_LTMean', scale=30, region=area_of_interest)

We can see that there are some gradients -- temperatures and turbidities change throughout the catchment. Some of that is linked to hydropower projects or other water diversion! We can also look at trends or differences in the data, for example by comparing average water temperatures or turbidities in the 1990s to those in the 2010s:

In [ ]:
temp_mean1 = water_collection.filterDate('1990-01-01', '2000-01-01').filterBounds(area_of_interest).select('surface_temp').reduce(ee.Reducer.mean())
temp_mean2 = water_collection.filterDate('2010-01-01', '2020-01-01').filterBounds(area_of_interest).select('surface_temp').reduce(ee.Reducer.mean())
temp_dif = temp_mean1.subtract(temp_mean2)
run_export(temp_dif, 'epsg:4326', 'Trishuli_WaterTemp_1990_2010', scale=30, region=area_of_interest)

In [ ]:
ndti_mean1 = water_collection.filterDate('1990-01-01', '2000-01-01').filterBounds(area_of_interest).select('NDTI').reduce(ee.Reducer.mean())
ndti_mean2 = water_collection.filterDate('2010-01-01', '2020-01-01').filterBounds(area_of_interest).select('NDTI').reduce(ee.Reducer.mean())
ndti_dif = ndti_mean1.subtract(ndti_mean2)
run_export(ndti_dif, 'epsg:4326', 'Trishuli_NDTI_1990_2010', scale=30, region=area_of_interest)

I have added here a screenshot of those differences -- the data is also available on [Github](https://github.com/tasmi/Workshop_Kathmandu_Feb2026) if you want to explore yourself. You can also change the area of interest and look at the same data for a new region!

<img src="https://raw.githubusercontent.com/tasmi/Workshop_Kathmandu_Feb2026/main/Notebooks/Images/Trishuli_WaterTemp_Change.png" alt="" style="width: 600px;"/>

For example, the water temperature is generally a couple of degrees warmer in the 2010s than in the 1990s!

## Sampling-based Approach using Earth Engine

This long-term average and gridded approach can work quite well in some cases, but it has a few weaknesses. One large issue is that there are many gaps in the time series (e.g., due to clouds), which can bias *trends*. We can minimize some of the issues with data gaps by aggregating nearby pixels. For example, instead of looking at all pixels within a river, we can take the average over a 500 m stretch of the river. If water temperatures don't change very much over short stretches, this can improve our data quality a lot and minimize gaps and outliers.

To do this, however, we need sample points along the river. We can calculate those from a flow network. If you are interested, here is a good resource for calculating flow networks: [https://topotoolbox.wordpress.com/](https://topotoolbox.wordpress.com/)

For this exercise, we will use a provided stream network over the Trishuli. This can be found on Github.

Let's load in the stream network data first:


In [ ]:
#gdf = gpd.read_file('../GeoData/Trishuli_RiverFiltered.gpkg')
gdf = gpd.read_file('https://github.com/tasmi/Workshop_Kathmandu_Feb2026/raw/refs/heads/main/Notebooks/GeoData/Trishuli_RiverFiltered.gpkg')
gdf.plot()

We can use any of these individual points to extract a *time series* of data using our long-term Landsat collection. Let's first sample one individual point. We can use the same approach as for extracting rainfall data we used before:

In [ ]:
single_point = gdf.iloc[0]
single_point

In [ ]:
single_point.geometry.centroid.x

In [ ]:
xy = [single_point.geometry.centroid.x, single_point.geometry.centroid.y]
ee_point = ee.Geometry.Point(xy)
print(ee_point.getInfo())

In [ ]:
def export_time_series_data(collection, geometry, variable): #This is the same function as before, just only getting means for speed
    import pandas as pd
    
    def create_time_series(image):
        date = image.get('system:time_start')
        mean = image.reduceRegion(reducer=ee.Reducer.mean(), geometry=geometry).get(variable)
        ft = ee.Feature(None, {'date': ee.Date(date).format('Y/M/d'), 'mean':mean})
        return ft

    time_series = collection.map(create_time_series).getInfo()

    dates, means = [], []
    for f in time_series['features']:
        properties = f['properties']
        try:
            val = properties['mean']
            means.append(val)
            date = properties['date']
            dates.append(datetime.datetime.strptime(date,'%Y/%m/%d')) #Convert the date into something that Python recognizes
        except:
            pass
        
    return pd.Series(np.array(means), index=np.array(dates))

data_subset = water_collection.filterBounds(ee_point).filterDate('2020-01-01','2025-01-01')
surface_temp1 = export_time_series_data(data_subset, ee_point.buffer(100), 'surface_temp') #Small buffer, only one or two pixels
surface_temp2 = export_time_series_data(data_subset, ee_point.buffer(1000), 'surface_temp') #Big buffer, several pixels averaged together

If we look at different buffer distances, we can get more or less time points -- there is quite a lot of missing data if we focus only one one pixel! 

The data plots end up looking similar, but with more data in the 'bigger' buffer.

In [ ]:
surface_temp1.shape, surface_temp2.shape

In [ ]:
f, ax = plt.subplots(1)
surface_temp1.plot(ax=ax, label='Small Buffer')
surface_temp2.plot(ax=ax, label='Big Buffer')
ax.legend()

Let's do the same with the turbidity data:

In [ ]:
data_subset = water_collection.filterBounds(ee_point).filterDate('2020-01-01','2025-01-01')
turb1 = export_time_series_data(data_subset, ee_point.buffer(100), 'NDTI') #Small buffer, only one or two pixels
turb2 = export_time_series_data(data_subset, ee_point.buffer(1000), 'NDTI') #Big buffer, several pixels averaged together

In [ ]:
f, ax = plt.subplots(1)
turb1.plot(ax=ax, label='Small Buffer')
turb2.plot(ax=ax, label='Big Buffer')
ax.legend()

Here the differences are bigger -- the turbidity changes a lot more over a small spatial area than the temperature does. This makes sense -- water temperature can stay the same around the curve of a river, but turbidity will change a lot as more or less sediment is carried in the faster/slower parts of the river. 

Let's see whether this is something that shows up above and below a hydropower dam -- we can look at two different time series and compare them. I have circled the points I used here:

<img src="https://raw.githubusercontent.com/tasmi/Workshop_Kathmandu_Feb2026/main/Notebooks/Images/Above_Below.png" alt="" style="width: 600px;"/>

Just from looking at the long-term temperature mean we can see that things are different at these two points of the river!

In [ ]:
above_dam = ee.Geometry.Point(84.482511,27.929228).buffer(250)
below_dam = ee.Geometry.Point(84.507585,27.921397).buffer(250)

turb1 = export_time_series_data(data_subset, above_dam, 'NDTI')
turb2 = export_time_series_data(data_subset, below_dam, 'NDTI')

temp1 = export_time_series_data(data_subset, above_dam, 'surface_temp')
temp2 = export_time_series_data(data_subset, below_dam, 'surface_temp')

In [ ]:
f, (ax, ax2) = plt.subplots(2, figsize=(10,5))
turb1.plot(ax=ax, label='Above Dam')
turb2.plot(ax=ax, label='Below Dam')
ax.legend()
ax.set_ylabel('Turbidity')

temp1.plot(ax=ax2, label='Above Dam')
temp2.plot(ax=ax2, label='Below Dam')
ax2.legend()
ax2.set_ylabel('Water Surface Temperature')

We can see that turbidity is generally higher above the dam and temperature is generally lower -- this fits with slow, deep water above the dam. If we look carefully, we can see that the difference in temperature and turbidity is not the same throughout the year -- sometimes they are more similar and sometimes more different. We can look at this month by month for more insight:

In [ ]:
turb_dif = turb1 - turb2
temp_dif = temp1 - temp2

f, (ax, ax2) = plt.subplots(2, figsize=(10,8))
ax.plot(turb_dif.index.dayofyear, turb_dif.values, '.')
ax.set_ylabel('Turbidity Difference')
ax.axhline(0,0,1, color='k', linestyle='--')

ax2.plot(temp_dif.index.dayofyear, temp_dif.values, '.')
ax2.set_ylabel('Water Surface Temperature Difference')
ax2.axhline(0,0,1, color='k', linestyle='--')
ax2.set_xlabel('Day of Year')

Differences are largest during the spring when cool water is collected in the reservoir above the dam. During the monsoon, the dam floodgates are generally open and the difference between the water above and below the dam is much smaller. 

If we only look at long-term means, we miss this kind of detailed information. To get a really good picture of what is happening to rivers, it is important to look at both maps and time series. Fortunately, with Google Earth Engine, we are able to do both!